# AgroGuard AI - Model Training Notebook

Fine-tunes MobileNetV2 for plant disease classification and compares three models on
**lab images** (PlantVillage) and **real field images** (PlantDoc).

| Model | Training data |
|---|---|
| Ready-made baseline | PlantVillage only (downloaded from Hugging Face, not trained here) |
| Model 2 | PlantVillage only (lab images) |
| Model 3 | PlantVillage + PlantDoc (lab + field images) |

**How to run (Google Colab):**
1. Runtime -> Change runtime type -> T4 GPU
2. Run all cells from top to bottom (the last cell trains both models, about 2-5 hours)
3. Results, plots and models are saved to Google Drive in the folder `hos052_agroguard`
   (if Colab disconnects, run the last cell again, it resumes from the last checkpoint).

**Datasets (not stored in GitHub):**
- PlantVillage: Hugging Face `BrandonFors/Plant-Diseases-PlantVillage-Dataset`
- PlantDoc: https://github.com/pratikkayal/PlantDoc-Dataset (CC BY 4.0)

The trained Model 3 weights are not stored in GitHub either (see the README for the download link).

## 1. Check the GPU

In [ ]:
# Check that the GPU is active
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## 2. Download the PlantVillage dataset (lab images)

In [ ]:
# Download the PlantVillage dataset from Hugging Face (takes a few minutes)
from datasets import load_dataset
import matplotlib.pyplot as plt
import random

dataset = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset")
print(dataset)

# Read the class names and count them
class_names = dataset["train"].features["label"].names
print("Number of classes:", len(class_names))

# Show 8 random training images with their labels
indices = random.sample(range(len(dataset["train"])), 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), indices):
    sample = dataset["train"][idx]
    ax.imshow(sample["image"])
    ax.set_title(class_names[sample["label"]].replace("___", "\n"), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("sample_images.png", dpi=150)
plt.show()

## 3. Download the PlantDoc dataset (real field images)

In [ ]:
# Download the PlantDoc dataset (real field images) from GitHub
import os

if not os.path.exists("PlantDoc-Dataset"):
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git

# Show the folder structure and number of images in each class folder
for split in sorted(os.listdir("PlantDoc-Dataset")):
    split_path = os.path.join("PlantDoc-Dataset", split)
    if os.path.isdir(split_path) and not split.startswith("."):
        print("\n==", split, "==")
        for cls in sorted(os.listdir(split_path)):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                print(f"  {cls}: {len(os.listdir(cls_path))} images")

## 4. Map PlantDoc class names to PlantVillage class names

In [ ]:
# Map each PlantDoc folder name to a PlantVillage class name using keywords
import os

# Each rule: PlantDoc folder -> (keyword 1, keyword 2) that must both appear
# in exactly ONE PlantVillage class name (compared in lowercase)
plantdoc_rules = {
    "Apple Scab Leaf": ("apple", "apple_scab"),
    "Apple leaf": ("apple", "healthy"),
    "Apple rust leaf": ("apple", "rust"),
    "Bell_pepper leaf": ("pepper", "healthy"),
    "Bell_pepper leaf spot": ("pepper", "bacterial"),
    "Blueberry leaf": ("blueberry", "healthy"),
    "Cherry leaf": ("cherry", "healthy"),
    "Corn Gray leaf spot": ("corn", "gray"),
    "Corn leaf blight": ("corn", "northern"),
    "Corn rust leaf": ("corn", "rust"),
    "Peach leaf": ("peach", "healthy"),
    "Potato leaf early blight": ("potato", "early"),
    "Potato leaf late blight": ("potato", "late"),
    "Raspberry leaf": ("raspberry", "healthy"),
    "Soyabean leaf": ("soybean", "healthy"),
    "Squash Powdery mildew leaf": ("squash", "powdery"),
    "Strawberry leaf": ("strawberry", "healthy"),
    "Tomato Early blight leaf": ("tomato", "early"),
    "Tomato Septoria leaf spot": ("tomato", "septoria"),
    "Tomato leaf": ("tomato", "healthy"),
    "Tomato leaf bacterial spot": ("tomato", "bacterial"),
    "Tomato leaf late blight": ("tomato", "late"),
    "Tomato leaf mosaic virus": ("tomato", "mosaic"),
    "Tomato leaf yellow virus": ("tomato", "yellow"),
    "Tomato mold leaf": ("tomato", "mold"),
    "grape leaf": ("grape", "healthy"),
    "grape leaf black rot": ("grape", "black_rot"),
}

plantdoc_to_pv = {}
for folder, (key1, key2) in plantdoc_rules.items():
    matches = [
        name for name in class_names
        if key1 in name.lower() and key2 in name.lower()
    ]
    if len(matches) == 1:
        plantdoc_to_pv[folder] = matches[0]
        print(f"OK       {folder:30s} -> {matches[0]}")
    else:
        print(f"PROBLEM  {folder:30s} -> matches: {matches}")

print("\nMapped classes:", len(plantdoc_to_pv), "of", len(plantdoc_rules))

## 5. Load the ready-made baseline and check that its class order matches the dataset

In [ ]:
# Load the ready-made model and compare its class order with our dataset
from transformers import AutoModelForImageClassification

MODEL_NAME = "linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification"
ready_model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)

print("Model classes:", len(ready_model.config.id2label))
print()
print(f"{'ID':<4} {'READY MODEL LABEL':<45} DATASET LABEL")
print("-" * 100)
for i in range(len(class_names)):
    print(f"{i:<4} {ready_model.config.id2label[i]:<45} {class_names[i]}")

## 6. Evaluate the ready-made baseline (the "before" result)

In [ ]:
# Evaluate the ready-made model on lab images (PlantVillage) and field images (PlantDoc)
import os
import json
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# Same preprocessing the MobileNetV2 model was trained with
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class HFImageDataset(Dataset):
    """Wraps a Hugging Face dataset split so PyTorch can read it."""

    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        image = item["image"].convert("RGB")
        return self.transform(image), item["label"]


class PlantDocDataset(Dataset):
    """Reads PlantDoc field images and converts folder names to PlantVillage class ids."""

    def __init__(self, root, mapping, class_names, transform):
        self.transform = transform
        self.samples = []
        name_to_id = {name: i for i, name in enumerate(class_names)}
        for folder, pv_name in mapping.items():
            folder_path = os.path.join(root, folder)
            if not os.path.isdir(folder_path):
                continue
            for file_name in sorted(os.listdir(folder_path)):
                if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(folder_path, file_name), name_to_id[pv_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            # Skip unreadable files by returning a blank image
            image = Image.new("RGB", (224, 224))
        return self.transform(image), label


def evaluate(model, loader):
    """Return accuracy plus all predictions and true labels."""
    model.eval()
    model.to(device)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(pixel_values=images.to(device)).logits
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())
    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    return correct / len(all_labels), all_preds, all_labels


# Build the two test sets
pv_test = HFImageDataset(dataset["test"], eval_transform)
pd_test = PlantDocDataset("PlantDoc-Dataset/test", plantdoc_to_pv, class_names, eval_transform)
print("PlantVillage test images:", len(pv_test))
print("PlantDoc (field) test images:", len(pd_test))

pv_loader = DataLoader(pv_test, batch_size=64, shuffle=False, num_workers=2)
pd_loader = DataLoader(pd_test, batch_size=64, shuffle=False, num_workers=2)

# Evaluate the ready-made model
acc_pv, _, _ = evaluate(ready_model, pv_loader)
acc_pd, _, _ = evaluate(ready_model, pd_loader)

print(f"\nREADY MODEL - PlantVillage test accuracy (lab images):  {acc_pv * 100:.2f}%")
print(f"READY MODEL - PlantDoc test accuracy (field images):    {acc_pd * 100:.2f}%")

# Save the numbers so we can use them in the report later
results = {"ready_model": {"plantvillage_test": acc_pv, "plantdoc_test": acc_pd}}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

## 7. Train Model 2 and Model 3, compare all models and save the plots

This cell trains both models (12 epochs each) with strong augmentation, evaluates after every epoch,
saves checkpoints to Google Drive, and creates the training curves, the comparison chart and the
confusion matrix.

In [ ]:
# ======================================================================
# AgroGuard AI - Train Model 2 and Model 3, compare all models, save plots
# Model 2: MobileNetV2 trained on PlantVillage only (lab images)
# Model 3: MobileNetV2 trained on PlantVillage + PlantDoc train (field images)
# Everything is saved to Google Drive, so a disconnect never loses progress.
# If Colab disconnects: just run this same cell again, it resumes automatically.
# ======================================================================
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from google.colab import drive
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from transformers import AutoModelForImageClassification
from sklearn.metrics import confusion_matrix

# ---------- Settings ----------
EPOCHS = 12
BATCH_SIZE = 64
LEARNING_RATE = 5e-4
PLANTDOC_REPEAT = 6  # PlantDoc train set is small, so it is repeated to matter

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/hos052_agroguard"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------- Strong augmentation (helps with messy real-world photos) ----------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


def build_model():
    """Create MobileNetV2 (ImageNet weights) with a new 38-class output layer."""
    return AutoModelForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224",
        num_labels=len(class_names),
        id2label={i: n for i, n in enumerate(class_names)},
        label2id={n: i for i, n in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )


def train_model(run_name, train_dataset):
    """Train one model. Saves a checkpoint to Google Drive after every epoch."""
    ckpt_path = f"{SAVE_DIR}/{run_name}_checkpoint.pt"
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    steps_per_epoch = len(train_dataset) // BATCH_SIZE
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * steps_per_epoch)
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    history, start_epoch = [], 0

    # Resume from the last checkpoint if one exists
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        history, start_epoch = ckpt["history"], ckpt["epoch"]
        print(f"[{run_name}] Resuming from epoch {start_epoch}")

    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        running_loss, start_time = 0.0, time.time()

        for step, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = criterion(model(pixel_values=images).logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()
            if step % 200 == 0:
                print(f"  [{run_name}] epoch {epoch + 1} step {step}/{steps_per_epoch} loss {loss.item():.4f}")

        # Evaluate on both test sets after each epoch
        acc_pv, _, _ = evaluate(model, pv_loader)
        acc_pd, _, _ = evaluate(model, pd_loader)
        history.append({
            "epoch": epoch + 1,
            "train_loss": running_loss / steps_per_epoch,
            "plantvillage_test": acc_pv,
            "plantdoc_test": acc_pd,
        })
        print(f"[{run_name}] Epoch {epoch + 1}/{EPOCHS} | loss {history[-1]['train_loss']:.4f} | "
              f"PlantVillage {acc_pv * 100:.2f}% | PlantDoc {acc_pd * 100:.2f}% | "
              f"{(time.time() - start_time) / 60:.1f} min")

        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "history": history,
                    "epoch": epoch + 1}, ckpt_path)

    model.save_pretrained(f"{SAVE_DIR}/{run_name}_final")
    with open(f"{SAVE_DIR}/{run_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model, history


# ---------- Datasets ----------
train_ds_pv = HFImageDataset(dataset["train"], train_transform)
train_ds_field = PlantDocDataset("PlantDoc-Dataset/train", plantdoc_to_pv, class_names, train_transform)
print("PlantVillage train images:", len(train_ds_pv))
print("PlantDoc (field) train images:", len(train_ds_field))
train_ds_mixed = ConcatDataset([train_ds_pv] + [train_ds_field] * PLANTDOC_REPEAT)

# ---------- Train Model 2 and Model 3 ----------
model2, history2 = train_model("v2_model2_plantvillage_only", train_ds_pv)
model3, history3 = train_model("v2_model3_plantvillage_plus_field", train_ds_mixed)

# ---------- Final results table ----------
results["model2_plantvillage_only"] = {
    "plantvillage_test": history2[-1]["plantvillage_test"],
    "plantdoc_test": history2[-1]["plantdoc_test"],
}
results["model3_plantvillage_plus_field"] = {
    "plantvillage_test": history3[-1]["plantvillage_test"],
    "plantdoc_test": history3[-1]["plantdoc_test"],
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n================ FINAL RESULTS ================")
print(f"{'Model':<40} {'PlantVillage':>14} {'PlantDoc':>10}")
for name, r in results.items():
    print(f"{name:<40} {r['plantvillage_test'] * 100:>13.2f}% {r['plantdoc_test'] * 100:>9.2f}%")

# ---------- Plot 1: training curves ----------
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for hist, label in [(history2, "Model 2 (PlantVillage only)"), (history3, "Model 3 (PlantVillage + field)")]:
    epochs_axis = [h["epoch"] for h in hist]
    axes[0].plot(epochs_axis, [h["train_loss"] for h in hist], marker="o", label=label)
    axes[1].plot(epochs_axis, [h["plantvillage_test"] * 100 for h in hist], marker="o", label=label)
    axes[2].plot(epochs_axis, [h["plantdoc_test"] * 100 for h in hist], marker="o", label=label)
axes[0].set_title("Training loss")
axes[1].set_title("Accuracy on lab images (PlantVillage test)")
axes[2].set_title("Accuracy on field images (PlantDoc test)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
axes[1].set_ylabel("Accuracy (%)")
axes[2].set_ylabel("Accuracy (%)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150)
plt.show()

# ---------- Plot 2: comparison bar chart ----------
labels_bar = ["Ready model", "Model 2\n(lab only)", "Model 3\n(lab + field)"]
keys = ["ready_model", "model2_plantvillage_only", "model3_plantvillage_plus_field"]
lab_scores = [results[k]["plantvillage_test"] * 100 for k in keys]
field_scores = [results[k]["plantdoc_test"] * 100 for k in keys]
x = np.arange(len(keys))
plt.figure(figsize=(8, 5))
plt.bar(x - 0.2, lab_scores, 0.4, label="Lab images (PlantVillage)")
plt.bar(x + 0.2, field_scores, 0.4, label="Field images (PlantDoc)")
plt.xticks(x, labels_bar)
plt.ylabel("Test accuracy (%)")
plt.title("Model comparison")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/model_comparison.png", dpi=150)
plt.show()

# ---------- Plot 3: confusion matrix of Model 3 on field images ----------
_, preds, true_labels = evaluate(model3, pd_loader)
used = sorted(set(true_labels))
cm = confusion_matrix(true_labels, preds, labels=used)
short_names = [class_names[i].replace("___", " - ")[:28] for i in used]
plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(used)), short_names, rotation=90, fontsize=7)
plt.yticks(range(len(used)), short_names, fontsize=7)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Model 3 confusion matrix on field images (PlantDoc test)")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/confusion_matrix_model3_field.png", dpi=150)
plt.show()

print("\nALL DONE. Files saved in Google Drive folder: hos052_agroguard")